## Financial Data Collection
This notebook downloads annual financial data from the SEC and builds **one clean table per company**
(one row per fiscal year). Then it combines all companies into one dataset.

**How it works**

```
1. Download company data from the SEC      -> get_facts()
2. Pull out each metric (Revenue, Assets)  -> get_metric()
3. Build one clean table for the company   -> build_company()
4. Check the table for mistakes            -> find_problems()
5. Repeat for every company, combine, save
```

**To add a company:** add its ticker to `TICKERS` in Step 2 and run the notebook again.**bold text**

## Rules the code follows

1. **Annual reports only** (form 10-K).
2. **One report per year.** A company can publish the same year several times and change the numbers
   (a *restatement*). If Assets came from one report and Equity from another, the balance sheet would not balance.
   So for each year we pick **one** report and take every number from it: the latest report that contains
   Assets, Liabilities and Equity together.
3. **Debt** = long-term debt (non-current + current part; if a company only reports one total, we use the total) + short-term borrowings. Leases are not included.
   A missing debt is **never** silently set to zero (see Step 7).
4. **CapEx** = cash spent on property and equipment. Finance leases are not included.
5. **Year** = the calendar year in which the fiscal year ends (Microsoft's year ends on June 30).

## Step 1: Imports

An *import* loads a ready-made toolbox so we do not write everything from scratch.

In [14]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import requests

## Step 2: Settings

The only place you need to edit when you add companies or change the start year.

In [15]:
SEC_USER_AGENT = "Rana Mamdoh your_email@example.com"   # the SEC requires your real name and email

TICKERS = ["MSFT","AAPL","GOOGL" , "META","AMZN"]
START_YEAR = 2017
ASSUME_NO_DEBT = {"META": [2017, 2018, 2019, 2020]}
RAW_FOLDER = Path("data/raw")
OUTPUT_FILE = Path("data/companies_annual.csv")


## Step 3: What we collect

Each metric has a list of SEC names (*tags*). Companies name the same thing differently, so we list a few options;
the first one has priority and the others are used only if the first is missing.

There are two kinds of metrics:
* **Yearly**: covers a whole year (like your salary for the year): Revenue, Net Profit, cash flows.
* **Balance sheet**: measured on one day, the last day of the year (like your bank balance on December 31): Assets, Liabilities, ...

In [16]:
YEARLY_TAGS = {
    "Revenue":           ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues", "SalesRevenueNet"],
    "NetProfit":         ["NetIncomeLoss"],
    "OperatingCashFlow": ["NetCashProvidedByUsedInOperatingActivities",
                          "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"],
    "CapEx":             ["PaymentsToAcquirePropertyPlantAndEquipment", "PaymentsToAcquireProductiveAssets"],
}

BALANCE_SHEET_TAGS = {
    "Assets":               ["Assets"],
    "Liabilities":          ["Liabilities"],
    "Equity":               ["StockholdersEquity",
                             "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "Cash":                 ["CashAndCashEquivalentsAtCarryingValue"],
    "ShortTermInvestments": ["ShortTermInvestments", "MarketableSecuritiesCurrent"],
    "DebtNoncurrent":       ["LongTermDebtNoncurrent"],
    "DebtCurrent":          ["LongTermDebtCurrent"],
    "DebtTotal":            ["LongTermDebt"],       # used only if the split above is missing
    "ShortTermDebt":        ["ShortTermBorrowings", "CommercialPaper"],
}

REQUIRED_COLUMNS = ["Revenue", "NetProfit", "OperatingCashFlow", "CapEx",
                    "Assets", "Liabilities", "Equity", "Cash", "Debt"]

## Step 4: download a company's data

A *function* is a saved recipe: you give it an input (a ticker) and it gives you back a result.

* `sec_request` sends a request to the SEC and returns the answer.
* `get_facts` finds the company, downloads all its data, and **saves it in a file**. The next time you ask for the same
  company it reads the file instead of downloading again.

In [17]:
def sec_request(url):
    time.sleep(0.2)   # the SEC allows at most 10 requests per second
    response = requests.get(url, headers={"User-Agent": SEC_USER_AGENT}, timeout=30)
    response.raise_for_status()
    return response.json()


def get_facts(ticker):
    RAW_FOLDER.mkdir(parents=True, exist_ok=True)
    saved_file = RAW_FOLDER / f"{ticker}_facts.json"

    if saved_file.exists():
        return json.loads(saved_file.read_text())

    companies = sec_request("https://www.sec.gov/files/company_tickers.json")
    cik = None
    for company in companies.values():
        if company["ticker"] == ticker:
            cik = int(company["cik_str"])
    if cik is None:
        raise ValueError(f"Ticker {ticker} was not found")

    facts = sec_request(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json")
    saved_file.write_text(json.dumps(facts))
    return facts

## Step 5: pull out one metric

The SEC data contains thousands of numbers. `get_metric` keeps only what we need for **one** metric:

1. Take the values under each of the metric's tags.
2. Keep annual reports only (`form == "10-K"`, `fp == "FY"`).
3. For yearly metrics, keep only values that cover about a year (350 to 380 days).
   Annual reports also contain quarterly numbers, and we do not want those.
4. If two tags exist in the same report, keep the one with priority.

The result is a small table with the columns `end` (the date the year ends), `accn` (the report's ID),
`filed` (when it was published) and `val` (the number).
The same year usually appears several times, once for every report that mentions it.

In [18]:
def get_metric(facts, tags, yearly):
    us_gaap = facts["facts"]["us-gaap"]

    tables = []
    for priority, tag in enumerate(tags):
        rows = us_gaap.get(tag, {}).get("units", {}).get("USD")
        if rows:
            table = pd.DataFrame(rows)
            table["priority"] = priority
            tables.append(table)

    if not tables:
        return pd.DataFrame(columns=["end", "accn", "filed", "val"])

    data = pd.concat(tables)
    data = data[(data["form"] == "10-K") & (data["fp"] == "FY")].copy()
    data["end"] = pd.to_datetime(data["end"])
    data["filed"] = pd.to_datetime(data["filed"])

    if yearly:
        days = (data["end"] - pd.to_datetime(data["start"])).dt.days
        data = data[(days >= 350) & (days <= 380)]

    data = data.sort_values("priority").drop_duplicates(["end", "accn"])
    return data[["end", "accn", "filed", "val"]]

## Step 6: build one company's table

This is the main function. It has three parts:

1. **Get every metric** using `get_metric`.
2. **Choose one report for each year:** the latest report that contains Assets, Liabilities *and* Equity.
   (`merge` keeps only the reports that appear in all three tables.)
3. **Take every metric from that same report**, then calculate `Debt`.

If a metric is missing in the chosen report, its cell stays empty (`NaN`) and the checks in the next step will tell you.

In [19]:
def build_company(ticker):
    facts = get_facts(ticker)
    tables = {}
    for name, tags in YEARLY_TAGS.items():
        tables[name] = get_metric(facts, tags, yearly=True)
    for name, tags in BALANCE_SHEET_TAGS.items():
        tables[name] = get_metric(facts, tags, yearly=False)

    if tables["Liabilities"].empty:
        total = get_metric(facts, ["LiabilitiesAndStockholdersEquity"], yearly=False)
        if not total.empty and not tables["Equity"].empty:
            derived = total.merge(tables["Equity"], on=["end", "accn"], suffixes=("", "_equity"))
            derived["val"] = derived["val"] - derived["val_equity"]
            tables["Liabilities"] = derived[["end", "accn", "filed", "val"]]
            print(f"{ticker}: no 'Total liabilities' line, so Liabilities = Total liabilities and equity - Equity")

    for name in ["Assets", "Liabilities", "Equity"]:
        if tables[name].empty:
            raise ValueError(f"{ticker}: no data for {name}. Use search_tags() at the end of the notebook to find its SEC name.")


    reports = (tables["Assets"][["end", "accn", "filed"]]
               .merge(tables["Liabilities"][["end", "accn"]], on=["end", "accn"])
               .merge(tables["Equity"][["end", "accn"]], on=["end", "accn"]))
    reports = reports.sort_values("filed").drop_duplicates("end", keep="last")
    df = reports.rename(columns={"end": "PeriodEnd", "accn": "Report", "filed": "SourceFiled"})

    DEBT_COLUMNS = {"DebtNoncurrent", "DebtCurrent", "DebtTotal", "ShortTermDebt"}

    for name, table in tables.items():
        if table.empty:
            df[name] = np.nan
            continue

        if name in DEBT_COLUMNS:
            # Debt is sometimes missing from the year's own 10-K but appears in a later
            # 10-K's comparative column. Debt is not part of Assets = Liabilities + Equity,
            # so we take the latest reported value for each period end, even from a
            # different filing than the one used for the balance sheet.
            latest = table.sort_values("filed").drop_duplicates("end", keep="last")
            latest = latest.rename(columns={"end": "PeriodEnd", "val": name})
            df = df.merge(latest[["PeriodEnd", name]], on="PeriodEnd", how="left")
        else:
            table = table.rename(columns={"end": "PeriodEnd", "accn": "Report", "val": name})
            df = df.merge(table[["PeriodEnd", "Report", name]], on=["PeriodEnd", "Report"], how="left")

    df["Year"] = df["PeriodEnd"].dt.year
    df = df[df["Year"] >= START_YEAR].sort_values("Year").reset_index(drop=True)

    print(f"{ticker}: {len(df)} years found, {df['Year'].min()} to {df['Year'].max()}")
    for name in tables:
        print(f"   {name:22s} {df[name].notna().sum()} of {len(df)} years")

    # Debt = long-term debt + short-term borrowings
    # long-term debt = non-current + current part; if the split is missing, use the company's reported total
    long_term_debt = df["DebtNoncurrent"] + df["DebtCurrent"].fillna(0)
    used_total = long_term_debt.isna() & df["DebtTotal"].notna()
    if used_total.any():
        print(f"   note: Debt uses the reported total (no split available) in {df.loc[used_total, 'Year'].tolist()}")
    long_term_debt = long_term_debt.fillna(df["DebtTotal"])
    df["Debt"] = long_term_debt + df["ShortTermDebt"].fillna(0)

    for year in ASSUME_NO_DEBT.get(ticker, []):
        df.loc[(df["Year"] == year) & df["Debt"].isna(), "Debt"] = 0

    return df[["Year", "PeriodEnd", "Revenue", "NetProfit", "Assets", "Liabilities", "Equity",
               "Cash", "ShortTermInvestments", "Debt", "OperatingCashFlow", "CapEx", "SourceFiled"]]

## Step 7: check for mistakes

Before we trust a company's table, we check four things.
The most important one is the accounting rule **Assets = Liabilities + Equity**. It must always be true.
If it is not, some numbers came from different reports.

The function returns a list of problems. An empty list means the table is good.

In [20]:
def find_problems(df):
    problems = []
    if df["Year"].duplicated().any():
        problems.append(f"duplicate years: {df.loc[df['Year'].duplicated(), 'Year'].tolist()}")
    missing_years = sorted(set(range(df["Year"].min(), df["Year"].max() + 1)) - set(df["Year"]))
    if missing_years:
        problems.append(f"missing years: {missing_years}")

    for column in REQUIRED_COLUMNS:
        empty_years = df.loc[df[column].isna(), "Year"].tolist()
        if empty_years:
            hint = ""
            if column == "Debt":
                first_year_with_debt = df.loc[df["Debt"].notna(), "Year"].min()
                if empty_years and max(empty_years) < first_year_with_debt:
                    hint = (" <- all BEFORE the first year with debt data. Maybe the company had no debt: "
                            "check the 10-K, then use ASSUME_NO_DEBT in Step 2")
                else:
                    hint = " <- a gap between years with data. Probably another SEC tag: run explain_failed()"
            problems.append(f"{column} is empty in {empty_years}{hint}")

    gap = df["Assets"] - df["Liabilities"] - df["Equity"]
    bad_years = df.loc[gap.abs() > 0.001 * df["Assets"], "Year"].tolist()
    if bad_years:
        problems.append(f"Assets != Liabilities + Equity in {bad_years}")

    return problems

## Step 8: Run for every company

For each ticker we build the table and check it.
Companies that pass go into `datasets`. Companies with problems go into `failed`, with the problems printed,
so you can look at them and fix them.

In [21]:
datasets = {}
failed = {}
errors = {}

for ticker in TICKERS:
    try:
        df = build_company(ticker)
    except Exception as error:
        errors[ticker] = str(error)
        print(f"\n>>> {ticker} ERROR: {error}")
        continue
    problems = find_problems(df)

    if problems:
        failed[ticker] = df
        print(f"\n>>> {ticker} FAILED the checks:")
        for problem in problems:
            print("    -", problem)

    else:
        datasets[ticker] = df
        print(f">>> {ticker}: all checks passed\n")


MSFT: 10 years found, 2017 to 2026
   Revenue                10 of 10 years
   NetProfit              10 of 10 years
   OperatingCashFlow      10 of 10 years
   CapEx                  10 of 10 years
   Assets                 10 of 10 years
   Liabilities            10 of 10 years
   Equity                 10 of 10 years
   Cash                   10 of 10 years
   ShortTermInvestments   9 of 10 years
   DebtNoncurrent         10 of 10 years
   DebtCurrent            10 of 10 years
   DebtTotal              10 of 10 years
   ShortTermDebt          6 of 10 years
>>> MSFT: all checks passed

AAPL: 9 years found, 2017 to 2025
   Revenue                9 of 9 years
   NetProfit              9 of 9 years
   OperatingCashFlow      9 of 9 years
   CapEx                  9 of 9 years
   Assets                 9 of 9 years
   Liabilities            9 of 9 years
   Equity                 9 of 9 years
   Cash                   9 of 9 years
   ShortTermInvestments   8 of 9 years
   DebtNoncurrent   

Look at every company's table. Money is shown in **billions of dollars** to make it easier to read.

In [22]:
def in_billions(df):
    money_columns = ["Revenue", "NetProfit", "Assets", "Liabilities", "Equity",
                     "Cash", "ShortTermInvestments", "Debt", "OperatingCashFlow", "CapEx"]
    table = df.copy()
    table[money_columns] = (table[money_columns] / 1e9).round(2)
    return table


from IPython.display import display

for name, table in list(datasets.items()) + list(failed.items()):
    print(name, "(passed)" if name in datasets else "(FAILED, see the problems above)")
    display(in_billions(table))

MSFT (passed)


,Year,PeriodEnd,Revenue,NetProfit,Assets,Liabilities,Equity,Cash,ShortTermInvestments,Debt,OperatingCashFlow,CapEx,SourceFiled
0,2017,2017-06-30,96.57,25.49,250.31,162.60,87.71,7.66,NaN,86.19,39.51,8.13,2018-08-03
1,2018,2018-06-30,110.36,16.57,258.85,176.13,82.72,11.95,121.82,76.24,43.88,11.63,2019-08-01
2,2019,2019-06-30,125.84,39.24,286.56,184.23,102.33,11.36,122.46,72.18,52.18,13.92,2020-07-30
3,2020,2020-06-30,143.01,44.28,301.31,183.01,118.30,13.58,122.95,63.33,60.68,15.44,2021-07-29
4,2021,2021-06-30,168.09,61.27,333.78,191.79,141.99,14.22,116.11,58.15,76.74,20.62,2022-07-28
5,2022,2022-06-30,198.27,72.74,364.84,198.30,166.54,13.93,90.83,49.78,89.04,23.89,2023-07-27
6,2023,2023-06-30,211.92,72.36,411.98,205.75,206.22,34.70,76.56,47.24,87.58,28.11,2024-07-30
7,2024,2024-06-30,245.12,88.14,512.16,243.69,268.48,18.32,57.23,51.63,118.55,44.48,2025-07-30
8,2025,2025-06-30,281.72,101.83,619.00,275.52,343.48,30.24,64.32,43.15,136.16,64.55,2026-07-29
9,2026,2026-06-30,331.84,133.75,758.38,315.99,442.39,20.94,55.91,40.29,182.94,115.95,2026-07-29


AAPL (passed)


,Year,PeriodEnd,Revenue,NetProfit,Assets,Liabilities,Equity,Cash,ShortTermInvestments,Debt,OperatingCashFlow,CapEx,SourceFiled
0,2017,2017-09-30,229.23,48.35,375.32,241.27,134.05,20.29,NaN,115.68,64.22,12.45,2018-11-05
1,2018,2018-09-29,265.60,59.53,365.72,258.58,107.15,25.91,40.39,114.48,77.43,13.31,2019-10-31
2,2019,2019-09-28,260.17,55.26,338.52,248.03,90.49,48.84,51.71,108.05,69.39,10.50,2020-10-30
3,2020,2020-09-26,274.52,57.41,323.89,258.55,65.34,38.02,52.93,112.44,80.67,7.31,2021-10-29
4,2021,2021-09-25,365.82,94.68,351.00,287.91,63.09,34.94,27.70,124.72,104.04,11.08,2022-10-28
5,2022,2022-09-24,394.33,99.80,352.76,302.08,50.67,23.65,24.66,120.07,122.15,10.71,2023-11-03
6,2023,2023-09-30,383.28,97.00,352.58,290.44,62.15,29.96,31.59,111.09,110.54,10.96,2024-11-01
7,2024,2024-09-28,391.04,93.74,364.98,308.03,56.95,29.94,35.23,106.63,118.25,9.45,2025-10-31
8,2025,2025-09-27,416.16,112.01,359.24,285.51,73.73,35.93,18.76,98.66,111.48,12.72,2025-10-31


GOOGL (passed)


,Year,PeriodEnd,Revenue,NetProfit,Assets,Liabilities,Equity,Cash,ShortTermInvestments,Debt,OperatingCashFlow,CapEx,SourceFiled
0,2017,2017-12-31,110.86,12.66,197.30,44.79,152.50,10.72,91.16,3.94,37.09,13.18,2019-02-05
1,2018,2018-12-31,136.82,30.74,232.79,55.16,177.63,16.70,92.44,3.95,47.97,25.14,2020-02-04
2,2019,2019-12-31,161.86,34.34,275.91,74.47,201.44,18.50,101.18,3.96,54.52,23.55,2021-02-03
3,2020,2020-12-31,182.53,40.27,319.62,97.07,222.54,26.46,110.23,15.32,65.12,22.28,2022-02-02
4,2021,2021-12-31,257.64,76.03,359.27,107.63,251.64,20.94,118.70,15.44,91.65,24.64,2023-02-03
5,2022,2022-12-31,282.84,59.97,365.26,109.12,256.14,21.88,91.88,15.31,91.50,31.48,2024-01-31
6,2023,2023-12-31,307.39,73.80,402.39,119.01,283.38,24.05,86.87,12.87,101.75,32.25,2025-02-05
7,2024,2024-12-31,350.02,100.12,450.26,125.17,325.08,23.47,72.19,14.18,125.30,52.54,2026-02-05
8,2025,2025-12-31,402.84,132.17,595.28,180.02,415.26,30.71,96.14,48.54,164.71,91.45,2026-02-05


META (passed)


,Year,PeriodEnd,Revenue,NetProfit,Assets,Liabilities,Equity,Cash,ShortTermInvestments,Debt,OperatingCashFlow,CapEx,SourceFiled
0,2017,2017-12-31,40.65,15.93,84.52,10.18,74.35,8.08,NaN,0.00,24.22,6.73,2019-01-31
1,2018,2018-12-31,55.84,22.11,97.33,13.21,84.13,10.02,NaN,0.00,29.27,13.92,2020-01-30
2,2019,2019-12-31,70.70,18.48,133.38,32.32,101.05,19.08,NaN,0.00,36.31,15.10,2021-01-28
3,2020,2020-12-31,85.96,29.15,159.32,31.03,128.29,17.58,NaN,0.00,38.75,15.12,2022-02-03
4,2021,2021-12-31,117.93,39.37,165.99,41.11,124.88,16.60,31.40,0.00,57.68,18.69,2023-02-02
5,2022,2022-12-31,116.61,23.20,185.73,60.01,125.71,14.68,26.06,9.92,50.48,31.43,2024-02-02
6,2023,2023-12-31,134.90,39.10,229.62,76.46,153.17,41.86,23.54,18.39,71.11,27.04,2025-01-30
7,2024,2024-12-31,164.50,62.36,276.05,93.42,182.64,43.89,33.93,28.83,91.33,37.26,2026-01-29
8,2025,2025-12-31,200.97,60.46,366.02,148.78,217.24,35.87,45.72,58.74,115.80,69.69,2026-01-29


AMZN (passed)


,Year,PeriodEnd,Revenue,NetProfit,Assets,Liabilities,Equity,Cash,ShortTermInvestments,Debt,OperatingCashFlow,CapEx,SourceFiled
0,2017,2017-12-31,177.87,3.03,131.31,103.60,27.71,20.52,10.46,24.84,18.36,11.96,2019-02-01
1,2018,2018-12-31,232.89,10.07,162.65,119.10,43.55,31.75,9.50,24.87,30.72,13.43,2020-01-31
2,2019,2019-12-31,280.52,11.59,225.25,163.19,62.06,36.09,18.93,24.72,38.51,16.86,2021-02-03
3,2020,2020-12-31,386.06,21.33,321.20,227.79,93.40,42.12,42.27,32.97,66.06,40.14,2022-02-04
4,2021,2021-12-31,469.82,33.36,420.55,282.30,138.24,36.22,59.83,50.24,46.33,61.05,2023-02-03
5,2022,2022-12-31,513.98,-2.72,462.68,316.63,146.04,53.89,16.14,70.15,46.75,63.64,2024-02-02
6,2023,2023-12-31,574.78,30.42,527.85,325.98,201.88,73.39,13.39,66.96,84.95,52.73,2025-02-07
7,2024,2024-12-31,637.96,59.25,624.89,338.92,285.97,78.78,22.42,57.79,115.88,83.00,2026-02-06
8,2025,2025-12-31,716.92,77.67,818.04,406.98,411.06,86.81,36.22,68.85,139.51,131.82,2026-02-06


### Extra: why is a value empty?

For every company that failed, `explain_failed()` lists the SEC tags that have data in the empty years.
That tag is usually the name you need to add in Step 3.

In [23]:
SEARCH_WORDS = {
    "Revenue":           ["Revenue", "Sales"],
    "NetProfit":         ["NetIncome"],
    "OperatingCashFlow": ["OperatingActivities"],
    "CapEx":             ["PaymentsToAcquire", "Capital"],
    "Cash":              ["Cash"],
    "Debt":              ["Debt", "Notes", "Borrowings", "Commercial"],
}


def explain_failed():
    """For every failed company: which SEC tags have data in the years that are empty?"""
    for ticker, df in failed.items():
        facts = get_facts(ticker)
        print(f"\n===== {ticker} =====")
        for column in REQUIRED_COLUMNS:
            empty_years = df.loc[df[column].isna(), "Year"].tolist()
            if not empty_years or column not in SEARCH_WORDS:
                continue
            print(f"{column} is empty in {empty_years}. SEC tags with data in those years:")
            yearly = column in YEARLY_TAGS
            found = False
            for tag in facts["facts"]["us-gaap"]:
                if any(word.lower() in tag.lower() for word in SEARCH_WORDS[column]):
                    table = get_metric(facts, [tag], yearly=yearly)
                    if table.empty:
                        continue
                    years = sorted(set(table["end"].dt.year) & set(empty_years))
                    if years:
                        print(f"    {tag}  ->  {years}")
                        found = True
            if not found:
                print("  (no tag found. Check the 10-K, or look with search_tags())")


explain_failed()

## Step 9: Combine all companies and save

We add a `Ticker` column to each company's table, then stack the tables on top of each other.

In [24]:
if not datasets:
    print("Nothing to save: no company passed the checks. Look at the problems printed in Step 8.")
else:
    all_companies = pd.concat(
        [df.assign(Ticker=ticker) for ticker, df in datasets.items()],
        ignore_index=True,
    )
    all_companies = all_companies[["Ticker"] + [c for c in all_companies.columns if c != "Ticker"]]

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    all_companies.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved {len(datasets)} companies ({len(all_companies)} rows) to {OUTPUT_FILE}")

if failed:
    print("NOT saved because they failed the checks:", list(failed))

Saved 5 companies (46 rows) to data/companies_annual.csv


## when a metric is missing for a company

Different companies sometimes use different SEC names for the same thing.
`search_tags` lists every SEC name that contains a word, so you can find the right one and add it to the tag lists in Step 3.

In [25]:
def search_tags(ticker, word):
    facts = get_facts(ticker)
    for tag in facts["facts"]["us-gaap"]:
        if word.lower() in tag.lower():
            print(tag)


## Download the CSV file

In [26]:
from google.colab import files
files.download("data/companies_annual.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>